# Homework PySpark Movie Reviews


In [19]:

!pip install -q pyspark

In [20]:
# download dataset
import kagglehub, os
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
csv_path = os.path.join(path, "IMDB Dataset.csv")
print(csv_path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, trim

spark = SparkSession.builder.appName("MovieReviews").getOrCreate()

## Load the dataset

In [22]:
df = spark.read.csv(csv_path, header=True, inferSchema=True, multiLine=True, escape='"')

## Explore the data

In [23]:
df.printSchema()

root
 |-- review: string (nullable = true)
 |-- sentiment: string (nullable = true)



In [24]:
df.show(5)

+--------------------+---------+
|              review|sentiment|
+--------------------+---------+
|One of the other ...| positive|
|A wonderful littl...| positive|
|I thought this wa...| positive|
|Basically there's...| negative|
|Petter Mattei's "...| positive|
+--------------------+---------+
only showing top 5 rows


## Clean the data

In [25]:
# remove null
clean_df = df.filter(col("review").isNotNull() & (trim(col("review")) != ""))

## Analyze the data

In [26]:
print("Total reviews:", clean_df.count())

Total reviews: 50000


In [27]:
# positive and negative count
clean_df.groupBy("sentiment").count().show()

+---------+-----+
|sentiment|count|
+---------+-----+
| positive|25000|
| negative|25000|
+---------+-----+



## Transform the data

In [28]:
# add review length
clean_df = clean_df.withColumn("review_length", length(col("review")))
clean_df.show(5)

+--------------------+---------+-------------+
|              review|sentiment|review_length|
+--------------------+---------+-------------+
|One of the other ...| positive|         1761|
|A wonderful littl...| positive|          998|
|I thought this wa...| positive|          926|
|Basically there's...| negative|          748|
|Petter Mattei's "...| positive|         1317|
+--------------------+---------+-------------+
only showing top 5 rows


In [29]:

long_reviews = clean_df.filter(col("review_length") > 500)
print("Reviews longer than 500:", long_reviews.count())
long_reviews.show(5)

Reviews longer than 500: 45017
+--------------------+---------+-------------+
|              review|sentiment|review_length|
+--------------------+---------+-------------+
|One of the other ...| positive|         1761|
|A wonderful littl...| positive|          998|
|I thought this wa...| positive|          926|
|Basically there's...| negative|          748|
|Petter Mattei's "...| positive|         1317|
+--------------------+---------+-------------+
only showing top 5 rows


## Save the results

In [30]:
clean_df.coalesce(1).write.csv("output/cleaned_reviews", header=True, mode="overwrite")
print("Saved")

Saved


In [31]:

!ls -lh output/cleaned_reviews

total 64M
-rw-r--r-- 1 root root 64M Nov 22 17:52 part-00000-38751195-b6c4-4aeb-9f5e-e8a1b1059e26-c000.csv
-rw-r--r-- 1 root root   0 Nov 22 17:52 _SUCCESS


In [32]:
spark.stop()